# درس ۰۳ - الگوهای طراحی عامل‌محور

در این درس، سه الگوی بنیادین طراحی برای ساخت عامل‌های هوش مصنوعی مؤثر را بررسی می‌کنیم:

۱. **دستورات روشن برای عامل** — طراحی دستورات دقیق و نقش‌تعریف‌کننده که رفتار عامل را هدایت می‌کند
۲. **خروجی ساختاریافته با مدل‌های Pydantic** — اطمینان از بازگشت داده‌های پیش‌بینی‌شده و اعتبارسنجی شده توسط عامل‌ها
۳. **عامل‌های با مسئولیت تک** — طراحی عامل‌های متمرکز که هر کدام یک کار را به خوبی انجام می‌دهند

هر الگو را روی سناریوی **پیشنهاد دهنده مقصد سفر** به کار می‌بریم و به تدریج سیستمی می‌سازیم که بتواند مقصد پیشنهاد دهد، در دسترس بودن را بررسی کند و امور لجستیکی را مدیریت نماید.


## راه‌اندازی


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## الگو ۱: دستورالعمل‌های واضح برای عامل

موثرترین الگو همچنین ساده‌ترین است: نوشتن دستورالعمل‌های واضح و دقیق برای عامل خود.

دستورالعمل‌های خوب مشخص می‌کنند:
- **چه کسی** عامل است (شخصیت و لحن)
- **چه کاری** باید انجام دهد (مسئولیت‌ها گام‌به‌گام)
- **چگونه** باید رفتار کند (محدودیت‌ها و سبک)

در ادامه، ما یک نماینده مسافرتی با دستورالعمل‌های صریح ایجاد می‌کنیم که هر پاسخی را که تولید می‌کند شکل می‌دهد.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## الگو ۲: خروجی ساختاریافته با مدل‌های Pydantic

متن آزاد برای گفتگو مفید است، اما سیستم‌های پس‌زمینه به داده‌های ساختاریافته نیاز دارند.
با جفت کردن **مدل‌های Pydantic** با یک **تابع ابزار**، می‌توانیم:

- تعریف یک طرح دقیق برای خروجی عامل
- اعتبارسنجی خودکار پاسخ‌ها
- ادغام مطمئن نتایج عامل در منطق برنامه

کلید اجرایی این است که `response_format` را هنگام اجرای عامل ارسال کنیم. این باعث می‌شود
مدل یک شیء اعتبارسنجی‌شده `TravelRecommendations` (موجود در `response.value`) برگرداند
به جای متن آزاد. ابزار `get_destination_details` نیز یک نوع `DestinationRecommendation` برمی‌گرداند،
بنابراین داده‌ها از ابتدا تا انتها ساختاریافته باقی می‌مانند.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## الگوی ۳: عوامل با مسئولیت واحد

وظایف پیچیده از تقسیم کار در بین چند عامل متمرکز بهره می‌برند، که هر کدام یک مسئولیت واحد دارند:

- یک **کارشناس مقصد** که درباره مکان‌ها و دسترسی‌ها اطلاع دارد
- یک **برنامه‌ریز لجستیک** که پروازها، هتل‌ها و برنامه سفر را مدیریت می‌کند

این مشابه اصل مهندسی نرم‌افزار *تفکیک نگرانی‌ها* است — هر عامل آسان‌تر می‌تواند به صورت مستقل آزمایش، نگهداری و بهبود یابد.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## خلاصه

در این درس، سه الگوی طراحی عامل‌محور را در سناریوی توصیه‌گر سفر به کار بردیم:

| الگو | ایده کلیدی | مزیت |
|---|---|---|
| **دستورالعمل‌های واضح** | مشخص کردن شخصیت، مسئولیت‌ها و محدودیت‌ها از ابتدا | رفتار منسجم و مطابق برند عامل |
| **خروجی ساختاریافته** | استفاده از مدل‌های Pydantic به عنوان قالب پاسخ | نتایج معتبر و قابل خواندن توسط ماشین |
| **مسئولیت واحد** | دادن یک وظیفه متمرکز به هر عامل | آسان‌تر کردن تست، نگهداری و ترکیب |

این الگوها به‌طور طبیعی ترکیب می‌شوند — می‌توانید دستورالعمل‌های واضح را با خروجی ساختاریافته درون یک عامل با مسئولیت واحد ترکیب کنید تا سیستم‌های مقاوم و آماده تولید بسازید.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
